# CN vs LMCI vs AD MRI Classification (`.nii.gz`)

该版本已从 CT/PNG 流程改为 MRI NIfTI 流程，并从二分类改为三分类：
- 递归读取三类目录中的 `.nii.gz`（或 `.nii`）
- `CN` 目录标记为正常（label=0）
- `LMCI` 目录标记为轻度认知障碍中间阶段（label=1）
- `ADNI1` 目录标记为阿尔兹海默症（label=2）
- 不再进行文件重命名
- 使用 3D CNN 完成训练与评估


In [ ]:
# 如果环境缺少依赖，请取消注释后执行：
# !pip install nibabel scipy scikit-learn tensorflow

import os
import random
import logging
from pathlib import Path
from datetime import datetime

os.environ["TF_ENABLE_XLA"] = "0"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/home/biolab_374/anaconda3/envs/dl-assignment --xla_gpu_enable_triton_gemm=false"

# 避免 XLA 触发 libdevice.10.bc 错误（需在 import tensorflow 前设置）
os.environ.setdefault('TF_XLA_FLAGS', '--tf_xla_auto_jit=0')
if 'XLA_FLAGS' not in os.environ and os.environ.get('CONDA_PREFIX'):
    os.environ['XLA_FLAGS'] = f"--xla_gpu_cuda_data_dir={os.environ['CONDA_PREFIX']}"

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# 双保险：禁用 TF XLA JIT
tf.config.optimizer.set_jit(False)
print("TF_ENABLE_XLA =", os.environ.get("TF_ENABLE_XLA"))
print("TF_XLA_FLAGS  =", os.environ.get("TF_XLA_FLAGS"))
print("XLA_FLAGS     =", os.environ.get("XLA_FLAGS"))
tf.config.optimizer.set_jit(False)

# 可选：按需申请显存，降低 OOM 风险
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)
print('TF_XLA_FLAGS:', os.environ.get('TF_XLA_FLAGS'))
print('XLA_FLAGS:', os.environ.get('XLA_FLAGS'))


In [ ]:
# ----------------------------
# Config
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 修改为你的真实路径
CN_DIR = Path('/home/biolab_374/PycharmProjects/alz_image_classification/BIOCB26/image/Diffusion 7T Preprocessed Recommended')
LMCI_DIR = Path('/home/biolab_374/PycharmProjects/alz_image_classification/BIOCB26/image/LMCI-output')
ADNI1_DIR = Path('/home/biolab_374/PycharmProjects/alz_image_classification/BIOCB26/image/ADNI-output')

CLASS_NAMES = ['CN', 'LMCI', 'AD']
CLASS_TO_LABEL = {'CN': 0, 'LMCI': 1, 'AD': 2}

TARGET_SHAPE = (128, 128, 128)   # (D, H, W)
BATCH_SIZE = 4
EPOCHS = 30
LEARNING_RATE = 1e-4

# 运行日志：以模型开始运行日期时间开头
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_DIR = Path.cwd() / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f'{RUN_TIMESTAMP}_data_loading.log'

logger = logging.getLogger('alz_mri_loader')
logger.setLevel(logging.INFO)
logger.handlers.clear()
file_handler = logging.FileHandler(LOG_FILE, encoding='utf-8')
file_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
logger.addHandler(file_handler)
logger.propagate = False
logger.info('Run started.')
print('Log file:', LOG_FILE)

assert CN_DIR.exists(), f'CN_DIR not found: {CN_DIR}'
assert LMCI_DIR.exists(), f'LMCI_DIR not found: {LMCI_DIR}'
assert ADNI1_DIR.exists(), f'ADNI1_DIR not found: {ADNI1_DIR}'


In [ ]:
# ----------------------------
# Recursive scan of NIfTI files
# ----------------------------
def list_nifti_files(root: Path):
    nii_gz = list(root.rglob('*.nii.gz'))
    nii = list(root.rglob('*.nii'))
    # 去重：避免 .nii.gz 被重复匹配
    all_files = sorted(set(nii_gz + [p for p in nii if not str(p).endswith('.nii.gz')]))
    return all_files

cn_files = list_nifti_files(CN_DIR)
lmci_files = list_nifti_files(LMCI_DIR)
adni_files = list_nifti_files(ADNI1_DIR)

print('CN files:', len(cn_files))
print('LMCI files:', len(lmci_files))
print('AD files:', len(adni_files))

records = []
for p in cn_files:
    records.append({'filepath': str(p), 'label': CLASS_TO_LABEL['CN'], 'group': 'CN'})
for p in lmci_files:
    records.append({'filepath': str(p), 'label': CLASS_TO_LABEL['LMCI'], 'group': 'LMCI'})
for p in adni_files:
    records.append({'filepath': str(p), 'label': CLASS_TO_LABEL['AD'], 'group': 'AD'})

df = pd.DataFrame(records)
print(df['group'].value_counts())
df.head()


In [ ]:
# ----------------------------
# Train/Val/Test split
# ----------------------------
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df['label']
)

print('Train:', len(train_df), train_df['label'].value_counts().to_dict())
print('Val  :', len(val_df), val_df['label'].value_counts().to_dict())
print('Test :', len(test_df), test_df['label'].value_counts().to_dict())


In [ ]:
# ----------------------------
# NIfTI load + preprocess
# 改进点：如果文件损坏，记录到日志并跳过，不中断训练
# ----------------------------
BAD_FILES = []
BAD_FILE_PATHS = set()


def load_nifti_volume(path: str, target_shape=TARGET_SHAPE):
    img = nib.load(path)
    vol = img.get_fdata(dtype=np.float32)

    # 兼容 4D 数据，默认取第一个时间点/通道
    if vol.ndim == 4:
        vol = vol[..., 0]
    if vol.ndim != 3:
        raise ValueError(f'Unsupported volume shape: {vol.shape} @ {path}')

    # 强度归一化（对非零脑区做 robust min-max）
    mask = vol > 0
    if np.any(mask):
        vox = vol[mask]
        lo, hi = np.percentile(vox, [1, 99])
        vol = np.clip(vol, lo, hi)
        vol = (vol - lo) / (hi - lo + 1e-8)
        vol[~mask] = 0.0
    else:
        vol = np.zeros_like(vol, dtype=np.float32)

    # 重采样到固定形状
    factors = [t / s for t, s in zip(target_shape, vol.shape)]
    vol = zoom(vol, factors, order=1)

    # 添加通道维度: (D,H,W,1)
    vol = vol[..., np.newaxis].astype(np.float32)
    return vol


def safe_load_nifti_volume(path: str, label: int, group: str):
    try:
        volume = load_nifti_volume(path, TARGET_SHAPE)
        return volume, int(label)
    except Exception as e:
        if path not in BAD_FILE_PATHS:
            error_message = f'Failed to load NIfTI | group={group} | label={label} | path={path} | error={e}'
            logger.error(error_message)
            BAD_FILES.append({'filepath': path, 'label': int(label), 'group': group, 'error': str(e)})
            BAD_FILE_PATHS.add(path)
        return None


def dataset_generator(input_df, training=False):
    working_df = input_df.sample(frac=1, random_state=SEED).reset_index(drop=True) if training else input_df.reset_index(drop=True)

    for row in working_df.itertuples(index=False):
        result = safe_load_nifti_volume(row.filepath, row.label, row.group)
        if result is None:
            continue
        yield result


def make_dataset(input_df, batch_size=BATCH_SIZE, training=False):
    output_signature = (
        tf.TensorSpec(shape=TARGET_SHAPE + (1,), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )

    ds = tf.data.Dataset.from_generator(
        lambda: dataset_generator(input_df, training=training),
        output_signature=output_signature,
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)


In [ ]:
# ----------------------------
# 可视化样本（中间切片）
# ----------------------------
sample_row = None
vol = None
for row in train_df.itertuples(index=False):
    result = safe_load_nifti_volume(row.filepath, row.label, row.group)
    if result is not None:
        vol, sample_label = result
        sample_path = row.filepath
        sample_row = row
        break

if vol is None:
    raise RuntimeError(f'No valid training volume could be loaded. Check log: {LOG_FILE}')

mid_d = vol.shape[0] // 2
mid_h = vol.shape[1] // 2
mid_w = vol.shape[2] // 2

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(vol[mid_d, :, :, 0], cmap='gray')
ax[0].set_title('Axial')
ax[1].imshow(vol[:, mid_h, :, 0], cmap='gray')
ax[1].set_title('Coronal')
ax[2].imshow(vol[:, :, mid_w, 0], cmap='gray')
ax[2].set_title('Sagittal')
for a in ax:
    a.axis('off')
plt.suptitle(f'Label={int(sample_label)} | {Path(sample_path).name}')
plt.tight_layout()
plt.show()


In [ ]:
# ----------------------------
# 3D CNN model
# ----------------------------
def build_3d_cnn(input_shape=TARGET_SHAPE + (1,)):
    model = keras.Sequential([
        keras.layers.Input(shape=input_shape),
        keras.layers.Conv3D(16, 3, padding='same', activation='relu'),
        keras.layers.MaxPool3D(pool_size=2),

        keras.layers.Conv3D(32, 3, padding='same', activation='relu'),
        keras.layers.MaxPool3D(pool_size=2),

        keras.layers.Conv3D(64, 3, padding='same', activation='relu'),
        keras.layers.MaxPool3D(pool_size=2),

        keras.layers.GlobalAveragePooling3D(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(len(CLASS_NAMES), activation='softmax')
    ])
    return model

model = build_3d_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    jit_compile=False
)
model.summary()


In [ ]:
# ----------------------------
# Train
# ----------------------------
classes = np.sort(train_df['label'].unique())
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['label'].values
)
class_weight = {int(cls): float(weight) for cls, weight in zip(classes, class_weights_arr)}
print('class_weight:', class_weight)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=6, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('mri_3dcnn_best.keras', monitor='val_accuracy', mode='max', save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# ----------------------------
# Evaluate
# ----------------------------
test_metrics = model.evaluate(test_ds, verbose=1)
print('Test metrics:', dict(zip(model.metrics_names, test_metrics)))

# 预测与报告
y_true = test_df['label'].values
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:', confusion_matrix(y_true, y_pred))


In [ ]:
# ----------------------------
# Plot training curves
# ----------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Accuracy')

plt.tight_layout()
plt.show()


## Notes
- 本 notebook 已去掉原始工程中的批量重命名逻辑。
- 输入数据直接来自递归扫描的 `.nii.gz/.nii`。
- 如果显存不足，可降低 `TARGET_SHAPE`（如 `96,96,96`）和 `BATCH_SIZE`。
- 当前版本已从二分类改为三分类：CN / LMCI / AD。
- LMCI 被定义为 CN 和 AD 之间的中间阶段。
- 若要继续使用 EWC，可在本 3D CNN 的训练阶段叠加 EWC 正则项。当前版本先保证 MRI 主流程稳定可用。
